# 02 – Preprocessing Pipeline and Baseline

This notebook covers: cleaning, the train/test split and cross-validation design, the preprocessing pipeline, and the baseline results.

## 1. Load and clean

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")
df = df.drop_duplicates()             #12 exact duplicates 
df = df.drop(columns="duration")      #known only after the call -> leakage 

# y is coded 1 = "yes" (subscribed), 0 = "no".
X = df.drop(columns="y")
y = (df["y"] == "yes").astype(int)
print(X.shape, y.shape)

(41176, 19) (41176,)


## 2. Train/test split and cross-validation design

- Hold-out test set: 20% of the rows. Stratified so that both parts keep the same 11% of "yes" 
- Cross-validation: 5-fold StratifiedKFold on the training set only. All models are compared on these same folds.
- test set is not used until the final model has been chosen. It is used once, at the end.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "yes-rate:", round(y_train.mean(), 3))
print("Test: ", X_test.shape, "yes-rate:", round(y_test.mean(), 3))

Train: (32940, 19) yes-rate: 0.113
Test:  (8236, 19) yes-rate: 0.113


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. Preprocessing pipeline

- Numeric columns: StandardScaler -> puts every number on the same scale. Needed for Logistic regression.
- Categorical columns: OneHotEncoder -> makes one 0/1 column per category, including "unknown". handle_unknown="ignore" stops model from crashing on a category it never saw in training.

scaler and encoder learn from the data (means, standard deviations, the list of categories). Inside a pipeline they are re-fitted on the training folds only, so the validation fold and the test set don't influence them -> prevents data leakage

In [ ]:
#ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

## 4. Baseline models

- Dummy classifier: always predicts the most common class ("no") -> Any real model must beat it.
- Logistic regression with default settings:  simple learned baseline.

- F1 is the main metric:It balances precision and recall for the "yes" class and, unlike accuracy, is not affected by the imbalance.
- Secondary metrics: ROC-AUC, precision, recall and accuracy.


In [ ]:
#(max_iter=1000 so the solver converges).
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

models = {
    "Dummy (most frequent)": make_pipeline(prep, DummyClassifier(strategy="most_frequent")),
    "Logistic regression": make_pipeline(prep, LogisticRegression(max_iter=1000)),
}

In [ ]:
#reporting mean + standard deviation over the 5 folds.
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import cross_validate

metrics = {
    "f1": "f1",
    "roc_auc": "roc_auc",
    "precision": make_scorer(precision_score, zero_division=0),  # see note above
    "recall": "recall",
    "accuracy": "accuracy",
}
results = {}
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=metrics)
    results[name] = {m: f"{scores['test_' + m].mean():.3f} ± {scores['test_' + m].std():.3f}" for m in metrics}

pd.DataFrame(results).T

,f1,roc_auc,precision,recall,accuracy
Dummy (most frequent),0.000 ± 0.000,0.500 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.887 ± 0.000
Logistic regression,0.340 ± 0.004,0.790 ± 0.007,0.661 ± 0.025,0.229 ± 0.004,0.900 ± 0.001


## 5. Baseline results

(5-fold cross-validation on the training set, mean ± standard deviation; the test set has not been used.)

- The dummy is about 89% accurate but has F1 = 0 and recall = 0: never finds a single subscriber -> accuracy is misleading on imbalanced data.
- Logistic regression reaches an F1 of about 0.34 and a ROC-AUC of about 0.79, better than the dummy. The small standard deviations show results are stable across folds.
- Its recall is low (about 0.23): most subscribers are still missed.